<h1><center> New York Times API <br><br>
    <center> Data Collection <br><br>
    <center> Wendy Shi


# TOC

- [Main Query](#Main-Querying)
- [Single Query: Debugging](#Single-Query:-Debugging)
- [Merging Data](#Merging-Data)

# Preparation

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os

In [2]:
data = pd.read_excel("/Users/wendyshi2001/Documents/APIs/NYT_api.xlsx")
nyt_api_key = data.iloc[0,0]
#nyt_api_key

# Main Querying

### The Loop

In [26]:
#Used Chat-GPT to write me a function on how to scraped data on a daily basis
base_url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

start_date = datetime(2024, 10, 2)
end_date = datetime(2025, 1, 1)

all_articles = []
skipped_days = []

while start_date < end_date:
    begin_str = start_date.strftime('%Y%m%d')
    next_day_str = (start_date + timedelta(days=1)).strftime('%Y%m%d')
    print(f"\n📅 Checking {begin_str}...")

    # Step 1: Check number of hits
    check_params = {
        'q': 'japan',
        'begin_date': begin_str,
        'end_date': next_day_str,
        'page': 0,
        'api-key': nyt_api_key,
        #'fq': 'headline:("China")'
    }

    check_resp = requests.get(base_url, params=check_params)
    check_data = check_resp.json()
    hits = check_data.get("response", {}).get("metadata", {}).get("hits", 0)

    if hits == 0:
        print(f"📭 No results on {begin_str}")
        skipped_days.append(begin_str)
        start_date += timedelta(days=1)
        continue

    print(f"🔍 Found {hits} articles — fetching...")

    for page in range(0, min(10, (hits + 9) // 10)):  # max 10 pages/day
        params = {
            'q': 'japan',
            'begin_date': begin_str,
            'end_date': next_day_str,
            'page': page,
            'api-key': nyt_api_key
        }

        try:
            response = requests.get(base_url, params=params, timeout=10)
            data = response.json()
            docs = data.get('response', {}).get('docs', [])

            if not docs:
                break

            for doc in docs:
                all_articles.append({
                    'headline': doc['headline']['main'],
                    'abstract': doc.get('abstract'),
                    'pub_date': doc['pub_date'],
                    'section': doc.get('section_name'),
                    'url': doc['web_url'],
                    'query_date': begin_str,
                    'page_number': page
                })

        except Exception as e:
            print(f"⚠️ Error on {begin_str} page {page}: {e}")
            break

        time.sleep(20)  # polite pause

    start_date += timedelta(days=1)
    time.sleep(20)


📅 Checking 20241002...
🔍 Found 4 articles — fetching...

📅 Checking 20241003...
🔍 Found 10 articles — fetching...

📅 Checking 20241004...
🔍 Found 3 articles — fetching...

📅 Checking 20241005...
🔍 Found 1 articles — fetching...

📅 Checking 20241006...
🔍 Found 2 articles — fetching...

📅 Checking 20241007...
🔍 Found 4 articles — fetching...

📅 Checking 20241008...
🔍 Found 6 articles — fetching...

📅 Checking 20241009...
🔍 Found 6 articles — fetching...

📅 Checking 20241010...
🔍 Found 4 articles — fetching...

📅 Checking 20241011...
🔍 Found 11 articles — fetching...

📅 Checking 20241012...
📭 No results on 20241012

📅 Checking 20241013...
🔍 Found 3 articles — fetching...

📅 Checking 20241014...
🔍 Found 4 articles — fetching...

📅 Checking 20241015...
🔍 Found 3 articles — fetching...

📅 Checking 20241016...
🔍 Found 4 articles — fetching...

📅 Checking 20241017...
🔍 Found 9 articles — fetching...

📅 Checking 20241018...
🔍 Found 4 articles — fetching...

📅 Checking 20241019...
🔍 Found 1 art

In [32]:
### Convert to DataFrame
df = pd.DataFrame(all_articles)
print(df.shape)
df.head(20)

(476, 7)


,headline,abstract,pub_date,section,url,query_date,page_number
0,Here’s Where U.S. Forces Are Deployed in the M...,The Pentagon is preparing to send more troops ...,2024-10-02T07:32:29Z,World,https://www.nytimes.com/2024/10/02/world/middl...,20241002,0
1,"You Can Stand Under My Umbrella, if You’re an ...",Male locusts have long been observed shielding...,2024-10-02T09:02:13Z,Science,https://www.nytimes.com/2024/10/02/science/des...,20241002,0
2,Biden Says He Won’t Support an Israeli Strike ...,His comments reflected a renewed effort by his...,2024-10-02T18:05:32Z,World,https://www.nytimes.com/2024/10/02/world/middl...,20241002,0
3,Israel Says at Least 8 Soldiers Are Killed in ...,The cross-border fighting appeared to be the f...,2024-10-02T23:00:13Z,World,https://www.nytimes.com/2024/10/02/world/middl...,20241002,0
4,"In Japan’s Countryside, Century-Old Firms Lear...",Japan’s regional economies are facing severe l...,2024-10-03T04:00:14Z,Business,https://www.nytimes.com/2024/10/03/business/ja...,20241003,0
5,"Masamitsu Yoshioka, Last Pearl Harbor Bombardi...",He was 23 years old when he took part in the a...,2024-10-03T15:40:15Z,World,https://www.nytimes.com/2024/10/03/world/asia/...,20241003,0
6,"In Tokyo, a Small Gallery Focuses on Conceptua...","Striving to be unique, it is bringing attentio...",2024-10-03T09:02:53Z,Arts,https://www.nytimes.com/2024/10/03/arts/design...,20241003,0
7,Here are the latest developments.,,2024-10-03T08:46:09Z,World,https://www.nytimes.com/live/2024/10/03/world/...,20241003,0
8,A Recipe for a Striving America,The question is not whether to do industrial p...,2024-10-03T23:00:09Z,Opinion,https://www.nytimes.com/2024/10/03/opinion/ind...,20241003,0
9,How Tory Burch Gets Ready,"Plus: a new hotel on the Athens Riviera, salva...",2024-10-03T09:07:43Z,T Magazine,https://www.nytimes.com/2024/10/03/t-magazine/...,20241003,0


In [28]:
df.shape

(476, 7)

In [29]:
df.to_csv("Japan_Oct1Dec_2024.csv")

##

# Merging Data

## Chinese Data

In [3]:
df1 = pd.read_excel("Data/China_Jan.xlsx")
df1.head(3)

,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,May 3: China could launch the Chang’e-6 missio...,NaN,2024-01-01T17:15:25Z,Science,https://www.nytimes.com/article/may-china-coul...,20240101,0
1,1,China Auto Giant BYD Sells More Electric Vehic...,"Sales by BYD, the country’s dominant automaker...",2024-01-01T12:43:30Z,Business,https://www.nytimes.com/2024/01/01/business/by...,20240101,0
2,2,Tuesday Briefing: Israel’s Top Court Rejects M...,Plus New Year’s offerings to the sea in Brazil.,2024-01-01T20:41:37Z,Briefing,https://www.nytimes.com/2024/01/01/briefing/is...,20240101,0


In [4]:
df2 = pd.read_excel("Data/China_FebMar.xlsx")
df2.head(3)

,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,Coastal Cities Brace for Climate Change,This week’s atmospheric rivers may only be the...,2024-02-01T19:52:57Z,Climate,https://www.nytimes.com/2024/02/01/climate/coa...,20240201,0
1,1,Apple Sees First Quarterly Revenue Increase in...,The iPhone maker’s share price dropped in afte...,2024-02-01T22:05:44Z,Technology,https://www.nytimes.com/2024/02/01/technology/...,20240201,0
2,2,The Trump campaign is having South Carolina Re...,NaN,2024-02-01T16:11:26Z,U.S.,https://www.nytimes.com/live/2024/02/01/us/tru...,20240201,0


In [5]:
df3 = pd.read_excel("Data/China_AprMay.xlsx")
df3.head(3)

,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,China’s Advancing Efforts to Influence the U.S...,China has adopted some of the same misinformat...,2024-04-01T04:01:29Z,Business,https://www.nytimes.com/2024/04/01/business/me...,20240401,0
1,1,"Beijing Deplores Taiwan’s Next President, but ...",A rare visit to mainland China by Ma Ying-jeou...,2024-04-01T01:51:15Z,World,https://www.nytimes.com/2024/03/31/world/asia/...,20240401,0
2,2,"Protests Intensify Against Netanyahu, and Chin...","Plus, fast food workers get a raise.",2024-04-01T10:01:50Z,Podcasts,https://www.nytimes.com/2024/04/01/podcasts/is...,20240401,0


In [6]:
df4 = pd.read_excel("Data/China_June01_27.xlsx")
df4.head(3)

,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,Warnings of Election Meddling by China Never R...,A watchdog agency found roadblocks to the flow...,2024-06-01T10:00:05Z,World,https://www.nytimes.com/2024/06/01/world/canad...,20240601,0
1,1,Mixed News About Opioid Overdoses,Readers discuss reports of a decline in deaths...,2024-06-01T11:00:02Z,Opinion,https://www.nytimes.com/2024/06/01/opinion/opi...,20240601,0
2,2,Scandals and Missteps Slow Momentum of Germany...,The Alternative for Germany party remains stro...,2024-06-01T04:01:31Z,World,https://www.nytimes.com/2024/06/01/world/europ...,20240601,0


In [7]:
df5 = pd.read_excel("Data/China_June29_Sep.xlsx")
df5.head(3)

,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,“He gets paid by China.”,NaN,2024-06-28T02:33:05Z,U.S.,https://www.nytimes.com/live/2024/06/27/us/bid...,20240628,0
1,1,“We have the largest deficit with China.”,NaN,2024-06-28T02:33:16Z,U.S.,https://www.nytimes.com/live/2024/06/27/us/bid...,20240628,0
2,2,“The Paris accord was going to cost us $1 tril...,NaN,2024-06-28T02:21:22Z,U.S.,https://www.nytimes.com/live/2024/06/27/us/bid...,20240628,0


In [8]:
df6 = pd.read_excel("Data/China_Sep_1226.xlsx")
df6.head(3)

,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,China and the Philippines Trade Blame for the ...,The United States condemned the episode near S...,2024-09-01T09:30:15Z,World,https://www.nytimes.com/2024/09/01/world/asia/...,20240901,0
1,1,"Record Rainfall Spoils Crops in China, Rattlin...",Some vegetables cost more than they have in fi...,2024-09-02T06:06:41Z,World,https://www.nytimes.com/2024/09/02/world/asia/...,20240902,0
2,2,"Gao Zhen, Artist Who Critiqued the Cultural Re...",Mr. Gao is being held on suspicion of slanderi...,2024-09-02T11:32:41Z,World,https://www.nytimes.com/2024/09/02/world/asia/...,20240902,0


In [9]:
Full = pd.concat([df1, df2, df3, df4, df5, df6], axis = 0)
print(Full.shape)
Full.head(3)

(4241, 8)


,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,May 3: China could launch the Chang’e-6 missio...,NaN,2024-01-01T17:15:25Z,Science,https://www.nytimes.com/article/may-china-coul...,20240101,0
1,1,China Auto Giant BYD Sells More Electric Vehic...,"Sales by BYD, the country’s dominant automaker...",2024-01-01T12:43:30Z,Business,https://www.nytimes.com/2024/01/01/business/by...,20240101,0
2,2,Tuesday Briefing: Israel’s Top Court Rejects M...,Plus New Year’s offerings to the sea in Brazil.,2024-01-01T20:41:37Z,Briefing,https://www.nytimes.com/2024/01/01/briefing/is...,20240101,0


In [10]:
Full['abstract'] = Full['abstract'].fillna('none')

In [11]:
Full['headline'] = Full['headline'].astype(str)
Full['abstract'] = Full['abstract'].astype(str)
Full['Merged'] = Full['headline'] + ' ' + Full['abstract']
Full['Merged'] = Full['Merged'].str.strip()
Full.head(3)

,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number,Merged
0,0,May 3: China could launch the Chang’e-6 missio...,none,2024-01-01T17:15:25Z,Science,https://www.nytimes.com/article/may-china-coul...,20240101,0,May 3: China could launch the Chang’e-6 missio...
1,1,China Auto Giant BYD Sells More Electric Vehic...,"Sales by BYD, the country’s dominant automaker...",2024-01-01T12:43:30Z,Business,https://www.nytimes.com/2024/01/01/business/by...,20240101,0,China Auto Giant BYD Sells More Electric Vehic...
2,2,Tuesday Briefing: Israel’s Top Court Rejects M...,Plus New Year’s offerings to the sea in Brazil.,2024-01-01T20:41:37Z,Briefing,https://www.nytimes.com/2024/01/01/briefing/is...,20240101,0,Tuesday Briefing: Israel’s Top Court Rejects M...


In [12]:
Full.drop(columns = 'Unnamed: 0', inplace = True)
Full.head(3)

,headline,abstract,pub_date,section,url,query_date,page_number,Merged
0,May 3: China could launch the Chang’e-6 missio...,none,2024-01-01T17:15:25Z,Science,https://www.nytimes.com/article/may-china-coul...,20240101,0,May 3: China could launch the Chang’e-6 missio...
1,China Auto Giant BYD Sells More Electric Vehic...,"Sales by BYD, the country’s dominant automaker...",2024-01-01T12:43:30Z,Business,https://www.nytimes.com/2024/01/01/business/by...,20240101,0,China Auto Giant BYD Sells More Electric Vehic...
2,Tuesday Briefing: Israel’s Top Court Rejects M...,Plus New Year’s offerings to the sea in Brazil.,2024-01-01T20:41:37Z,Briefing,https://www.nytimes.com/2024/01/01/briefing/is...,20240101,0,Tuesday Briefing: Israel’s Top Court Rejects M...


In [15]:
# Compute string lengths
Full['length'] = Full['Merged'].str.split().apply(len)

# Get summary statistics
summary_stats = Full['length'].describe(percentiles=[0.25, 0.5, 0.75])[['min', '25%', '50%', '75%', 'max']]
print(summary_stats)

min     4.0
25%    23.0
50%    31.0
75%    35.0
max    60.0
Name: length, dtype: float64


In [12]:
import re

# Function to remove illegal characters for Excel
def clean_excel_text(text):
    return re.sub(r'[\x00-\x1F\x7F]', '', text)

# Apply to the 'Merged' column
Full['Merged'] = Full['Merged'].apply(clean_excel_text)

In [13]:
for col in Full.select_dtypes(include='object').columns:
    Full[col] = Full[col].apply(lambda x: re.sub(r'[\x00-\x1F\x7F]', '', str(x)))

In [14]:
Full.to_excel("China_merged_2024.xlsx")

## Japanese Data

In [16]:
df1 = pd.read_excel("Data/Japan_JanMar2024.xlsx")
print(df1.shape)
df1.head(3)

(416, 8)


,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,Japan’s meteorological agency warns of waves a...,Officials ordered residents of the Noto Penins...,2024-01-01T07:50:11Z,World,https://www.nytimes.com/live/2024/01/01/world/...,20240101,0
1,1,"Powerful Earthquake Hits Japan, and Officials ...",There were reports of collapsed buildings and ...,2024-01-01T19:06:41Z,World,https://www.nytimes.com/2024/01/01/world/asia/...,20240101,0
2,2,Map: Earthquake Strikes Japan,View the location of the quake’s epicenter and...,2024-01-01T14:08:08Z,World,https://www.nytimes.com/interactive/2024/01/01...,20240101,0


In [17]:
df2 = pd.read_excel("Data/Japan_AprJune2024.xlsx")
print(df2.shape)
df2.head(3)

(438, 8)


,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,"‘Oppenheimer’ Opens in Nuclear-Scarred Japan, ...",While some viewers lamented the movie’s exclus...,2024-04-01T06:03:08Z,World,https://www.nytimes.com/2024/04/01/world/asia/...,20240401,0
1,1,"In Kyoto, Five Hotels to Add to Your Travel Wi...",The city’s newest crop of hotels — from a luxu...,2024-04-01T21:03:38Z,T Magazine,https://www.nytimes.com/2024/04/01/t-magazine/...,20240401,0
2,2,"Lou Conter, Last Survivor of the Battleship Ar...",Escaping injury in the Japanese attack on the ...,2024-04-01T20:55:27Z,U.S.,https://www.nytimes.com/2024/04/01/us/lou-cont...,20240401,0


In [18]:
df3 = pd.read_csv("Data/Japan_JulySep6_2024.csv")
print(df3.shape)
df3.head(3)

(338, 8)


,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,Monday Briefing,Elections in France and Iran.,2024-07-01T04:52:55Z,Briefing,https://www.nytimes.com/2024/07/01/briefing/fr...,20240701,0
1,1,And the Winner Is … the Slowest!,Cargo ships off California are reducing speeds...,2024-07-02T09:02:51Z,Climate,https://www.nytimes.com/2024/07/02/climate/wha...,20240702,0
2,2,Cowboy Hats and Koi Fish Photos? There’s a Rea...,Some interior designers decorate their adult a...,2024-07-02T09:01:09Z,Real Estate,https://www.nytimes.com/2024/07/02/realestate/...,20240702,0


In [19]:
df4 = pd.read_csv("Data/Japan_Sep6Oct1_2024.csv")
print(df4.shape)
df4.head(3)

(144, 8)


,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,Japan Tries to Reclaim Its Clout as a Global T...,Japanese chip companies are tapping billions o...,2024-09-06T04:00:37Z,Business,https://www.nytimes.com/2024/09/06/business/ec...,20240906,0
1,1,U.S. Blowback to Steel Merger Vexes Japan,A looming decision in Washington to block Nipp...,2024-09-06T09:59:38Z,Business,https://www.nytimes.com/2024/09/06/business/ni...,20240906,0
2,2,7-Eleven Rejects Takeover Bid From Big Canadia...,"Japan’s Seven & i Holdings, the operator of 7-...",2024-09-06T03:40:39Z,Business,https://www.nytimes.com/2024/09/05/business/7-...,20240906,0


In [20]:
df5 = pd.read_csv("Data/Japan_Oct1Dec_2024.csv")
print(df5.shape)
df5.head(3)

(476, 8)


,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,Here’s Where U.S. Forces Are Deployed in the M...,The Pentagon is preparing to send more troops ...,2024-10-02T07:32:29Z,World,https://www.nytimes.com/2024/10/02/world/middl...,20241002,0
1,1,"You Can Stand Under My Umbrella, if You’re an ...",Male locusts have long been observed shielding...,2024-10-02T09:02:13Z,Science,https://www.nytimes.com/2024/10/02/science/des...,20241002,0
2,2,Biden Says He Won’t Support an Israeli Strike ...,His comments reflected a renewed effort by his...,2024-10-02T18:05:32Z,World,https://www.nytimes.com/2024/10/02/world/middl...,20241002,0


In [21]:
Full = pd.concat([df1, df2, df3, df4, df5], axis = 0)
print(Full.shape)
Full.head(3)

(1812, 8)


,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number
0,0,Japan’s meteorological agency warns of waves a...,Officials ordered residents of the Noto Penins...,2024-01-01T07:50:11Z,World,https://www.nytimes.com/live/2024/01/01/world/...,20240101,0
1,1,"Powerful Earthquake Hits Japan, and Officials ...",There were reports of collapsed buildings and ...,2024-01-01T19:06:41Z,World,https://www.nytimes.com/2024/01/01/world/asia/...,20240101,0
2,2,Map: Earthquake Strikes Japan,View the location of the quake’s epicenter and...,2024-01-01T14:08:08Z,World,https://www.nytimes.com/interactive/2024/01/01...,20240101,0


In [22]:
Full['headline'] = Full['headline'].astype(str)
Full['abstract'] = Full['abstract'].astype(str)
Full['abstract'] = Full['abstract'].fillna('none')
Full['Merged'] = Full['headline'] + ' ' + Full['abstract']
Full['Merged'] = Full['Merged'].str.strip()
Full.head(3)

,Unnamed: 0,headline,abstract,pub_date,section,url,query_date,page_number,Merged
0,0,Japan’s meteorological agency warns of waves a...,Officials ordered residents of the Noto Penins...,2024-01-01T07:50:11Z,World,https://www.nytimes.com/live/2024/01/01/world/...,20240101,0,Japan’s meteorological agency warns of waves a...
1,1,"Powerful Earthquake Hits Japan, and Officials ...",There were reports of collapsed buildings and ...,2024-01-01T19:06:41Z,World,https://www.nytimes.com/2024/01/01/world/asia/...,20240101,0,"Powerful Earthquake Hits Japan, and Officials ..."
2,2,Map: Earthquake Strikes Japan,View the location of the quake’s epicenter and...,2024-01-01T14:08:08Z,World,https://www.nytimes.com/interactive/2024/01/01...,20240101,0,Map: Earthquake Strikes Japan View the locatio...


In [23]:
Full.drop(columns = 'Unnamed: 0', inplace = True)
Full.head(3)

,headline,abstract,pub_date,section,url,query_date,page_number,Merged
0,Japan’s meteorological agency warns of waves a...,Officials ordered residents of the Noto Penins...,2024-01-01T07:50:11Z,World,https://www.nytimes.com/live/2024/01/01/world/...,20240101,0,Japan’s meteorological agency warns of waves a...
1,"Powerful Earthquake Hits Japan, and Officials ...",There were reports of collapsed buildings and ...,2024-01-01T19:06:41Z,World,https://www.nytimes.com/2024/01/01/world/asia/...,20240101,0,"Powerful Earthquake Hits Japan, and Officials ..."
2,Map: Earthquake Strikes Japan,View the location of the quake’s epicenter and...,2024-01-01T14:08:08Z,World,https://www.nytimes.com/interactive/2024/01/01...,20240101,0,Map: Earthquake Strikes Japan View the locatio...


In [24]:
# Compute string lengths
Full['length'] = Full['Merged'].str.split().apply(len)

# Get summary statistics
summary_stats = Full['length'].describe(percentiles=[0.25, 0.5, 0.75])[['min', '25%', '50%', '75%', 'max']]
print(summary_stats)

min     4.0
25%    22.0
50%    30.0
75%    35.0
max    53.0
Name: length, dtype: float64


In [49]:
import re

# Function to remove illegal characters for Excel
def clean_excel_text(text):
    return re.sub(r'[\x00-\x1F\x7F]', '', text)

# Apply to the 'Merged' column
Full['Merged'] = Full['Merged'].apply(clean_excel_text)

In [51]:
for col in Full.select_dtypes(include='object').columns:
    Full[col] = Full[col].apply(lambda x: re.sub(r'[\x00-\x1F\x7F]', '', str(x)))

In [52]:
Full.to_excel("Japan_merged_2024.xlsx")